## **Lab 08: Find Similar Items with PySpark MinHashLSH**

This notebook is a clean and complete working version of the lab.
It uses **PySpark**, **shingling**, **binary vectorization**, and **MinHashLSH** to find similar documents.

Why this version works reliably:
- it uses a small built-in dataset,
- it uses **word shingles** instead of long character shingles
- it uses a **distance threshold** that actually returns candidate pairs
- it removes self-matches and duplicate pairs at the end


## 1. Install and import the required libraries
Run this cell once in Colab. If you are using a local Jupyter environment with PySpark already installed, you can skip the installation lines.


In [1]:
# Uncomment these lines in Google Colab if needed
# !apt-get update -qq
# !apt-get install -y openjdk-17-jdk-headless -qq
# !pip install -q pyspark

import os
import re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, explode, collect_set, size, lit
from pyspark.ml.feature import CountVectorizer, MinHashLSH


## 2. Create a Spark session


In [ ]:
os.environ.setdefault("PYSPARK_PYTHON", "python3")

spark = SparkSession.builder     .appName("Lab08_Find_Similar_Items")     .master("local[*]")     .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


## 3. Build a small document collection
We intentionally include some clearly similar documents so that the notebook returns candidate pairs at the end.


In [ ]:
documents = [
    (0, "Space mission nasa rocket launch orbit moon satellite astronauts"),
    (1, "NASA rocket launch mission to the moon with astronauts and satellite"),
    (2, "Deep space telescope observes orbiting satellite near the moon"),
    (3, "Football team wins the league after a dramatic final match and goal"),
    (4, "Soccer team wins a final match after scoring a late goal in the league"),
    (5, "Basketball players train every day to improve defense offense and teamwork"),
    (6, "Machine learning models classify images using neural networks and features"),
    (7, "Deep learning neural networks analyze images and classify visual patterns"),
    (8, "Cooking pasta with tomato sauce garlic and cheese is simple and tasty"),
    (9, "Homemade pasta with tomato sauce and cheese is easy to cook"),
]

df = spark.createDataFrame(documents, ["id", "text"])
df.show(truncate=False)


## 4. Text preprocessing
We lowercase the text, remove non-alphanumeric characters, and normalize spaces.


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned_docs = [(doc_id, clean_text(text)) for doc_id, text in documents]
clean_df = spark.createDataFrame(cleaned_docs, ["id", "clean_text"])
clean_df.show(truncate=False)


## 5. Create shingles
We use **word 3-shingles**. This usually gives more meaningful overlap than long character shingles.


In [ ]:
def shingler(text, k=2):
    words = text.split()
    if len(words) < k:
        return [" ".join(words)] if words else []
    return [" ".join(words[i:i+k]) for i in range(len(words) - k + 1)]

shingled_docs = [(doc_id, shingler(text, k=2)) for doc_id, text in cleaned_docs]
shingle_df = spark.createDataFrame(shingled_docs, ["id", "shingles"])
shingle_df.show(truncate=False)


## 6. Explode shingles so Spark can vectorize them


In [ ]:
exploded = shingle_df.select("id", explode("shingles").alias("shingle"))
exploded.show(truncate=False)


In [36]:
exploded.count()

99

## 7. Group shingles back into arrays of unique shingles


In [ ]:
shingle_sets = exploded.groupBy("id").agg(collect_set("shingle").alias("shingle_set"))
shingle_sets.show(truncate=False)


## 8. Convert shingles into binary vectors
We use `CountVectorizer` with `binary=True` so that each document is represented as a binary presence/absence vector of shingles.


In [ ]:
cv = CountVectorizer(inputCol="shingle_set", outputCol="features", binary=True, minDF=1.0)
cv_model = cv.fit(shingle_sets)
binary_data = cv_model.transform(shingle_sets)

binary_data.select("id", "features").show(truncate=False)
print("Vocabulary size:", len(cv_model.vocabulary))


## 9. Apply MinHashLSH
Important: `approxSimilarityJoin` uses **Jaccard distance**, not Jaccard similarity.

- distance = 0 means very similar / identical
- distance = 1 means very different

So a threshold of `0.8` means we keep pairs with distance <= 0.8.


In [ ]:
mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=10)
lsh_model = mh.fit(binary_data)

hashed_data = lsh_model.transform(binary_data)
hashed_data.select("id", "hashes").show(truncate=False)


## 10. Find candidate document pairs
We remove self-matches and duplicate pairs, then convert Jaccard distance into Jaccard similarity.


In [ ]:
threshold = 0.9

similar_pairs = lsh_model.approxSimilarityJoin(
    hashed_data,
    hashed_data,
    threshold,
    distCol="JaccardDistance"
)

results = similar_pairs     .filter(col("datasetA.id") < col("datasetB.id"))     .select(
        col("datasetA.id").alias("Doc1"),
        col("datasetB.id").alias("Doc2"),
        col("JaccardDistance"),
        (lit(1.0) - col("JaccardDistance")).alias("JaccardSimilarity")
    )     .orderBy(col("JaccardSimilarity").desc())

results.show(50, truncate=False)


## 11. Join back the original text for easier interpretation


In [ ]:
left_docs = df.select(col("id").alias("Doc1"), col("text").alias("Text1"))
right_docs = df.select(col("id").alias("Doc2"), col("text").alias("Text2"))

final_results = results.join(left_docs, on="Doc1").join(right_docs, on="Doc2")
final_results.select("Doc1", "Doc2", "JaccardSimilarity", "Text1", "Text2").show(truncate=False)


## 12. Optional: test different thresholds
Try values such as `0.9`, `0.8`, `0.6`, and `0.4`.
A smaller threshold is stricter because it allows only smaller distances.


In [ ]:
for threshold in [0.9, 0.8, 0.6, 0.4]:
    pairs = lsh_model.approxSimilarityJoin(
        hashed_data,
        hashed_data,
        threshold,
        distCol="JaccardDistance"
    ).filter(col("datasetA.id") < col("datasetB.id"))

    count_pairs = pairs.count()
    print(f"Threshold = {threshold:.1f} -> candidate pairs = {count_pairs}")


##In this task, you will apply the same workflow used in the previous example, but now using the 20 Newsgroups dataset.

Steps to follow
Load the dataset
* Use the [20 Newsgroups dataset](https://github.com/gokriznastic/20-newsgroups_text-classification) and select a subset of documents, e.g, sci.space. Preprocess the text
* Convert the text to lowercase, remove punctuation, and clean the documents.
* Create shingles, Generate shingles for each document using the same idea as in the previous example.
* Transform the data into features, Convert the shingles into numerical feature vectors suitable for Spark.
*Apply MinHashLSH, Use Spark MinHashLSH to find candidate pairs of similar documents.
* Show the results, Display the detected document pairs together with their Jaccard distance and Jaccard similarity.
* Test different parameters, Repeat the experiment by changing:
the shingle size,
the distance threshold,
* Analyze the output, Answer the following questions:
Which document pair is the most similar?
How does changing the shingle size affect the results?
How does changing the threshold affect the number of candidate pairs?
Why is MinHashLSH more efficient than comparing all document pairs directly?

**Goal:** The objective is to verify that you understand how shingling, Jaccard similarity, and MinHashLSH work together to detect similar documents efficiently in a large text collection.


In [ ]:
from sklearn.datasets import fetch_20newsgroups
newsgroups = fetch_20newsgroups(subset='train', categories=['sci.space'], remove=('headers', 'footers', 'quotes'))

docs = newsgroups.data[:20]
doc_df = spark.createDataFrame([(i, d) for i, d in enumerate(docs)], ["id", "text"])
doc_df.show(truncate=100)

In [ ]:
# Stop Spark when finished
# spark.stop()
